![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 10 -- Lab 2: Homework Beat the Steam Task with Trees

Homework. On Day 08 a linear regression reached MAE 6.77 on the Steam game prices: the 50-point band, not full marks (below 6.6). Today's models can bend where a line cannot. You take the same features, try a tree, a forest and gradient boosting, tune the boosting with a grid search, read the forest's importances, and hand in a submission that should land in the full-mark band.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** `dataset_train.csv` (3,000 games) and `dataset_eval.csv` (600) from the ONIA repository, read from GitHub as on Day 08. The setup cell rebuilds Day 08's feature function (owners midpoint, release year, log reviews, positive share, one-hot genres).

---
# Setup

In [ ]:
# pandas, matplotlib and scikit-learn are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q pandas matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error

In [ ]:
BASE = "https://raw.githubusercontent.com/Olimpiada-AI/Propunere-probleme-si-solutii/main/Dataset%20si%20evaluator/Dataset/"
train = pd.read_csv(BASE + "dataset_train.csv")
test = pd.read_csv(BASE + "dataset_eval.csv")
print(train.shape, test.shape)

In [ ]:
genre_cols = list(train["Genres"].str.get_dummies(sep=",").columns)

def add_features(d):
    """Day 08 Lab 2 in one function, plus Recommendations; the genre columns are aligned to the training ones."""
    out = pd.DataFrame(index=d.index)
    for c in ["Metacritic score", "Positive", "Negative", "Recommendations"]:
        out[c] = d[c]
    parts = d["Estimated owners"].str.split(" - ")
    out["owners"] = (parts.str[0].astype(float) + parts.str[1].astype(float)) / 2
    out["year"] = pd.to_datetime(d["Release date"], format="mixed").dt.year
    out["log_positive"] = np.log1p(d["Positive"])
    out["positive_share"] = d["Positive"] / (d["Positive"] + d["Negative"] + 1)
    genres = d["Genres"].str.get_dummies(sep=",").reindex(columns=genre_cols, fill_value=0)
    return pd.concat([out, genres], axis=1)

F = add_features(train)
y = train["Price"]
F_train, F_val, y_train, y_val = train_test_split(F, y, test_size=0.2, random_state=42)
print(F_train.shape, F_val.shape)

---
# Part 1 -- The line to beat

## Task 1: The scorer and the linear baseline

Write `evaluate(model, cols)` that fits `model` on `F_train[cols], y_train`, prints the validation MAE with 2 decimals and returns it. Evaluate `make_pipeline(StandardScaler(), LinearRegression())` on `lin_cols = ["Metacritic score", "log_positive", "positive_share", "owners", "year"] + genre_cols` (Day 08's best set). Expected MAE 6.77. Store it in `results = {"linear": ...}`. Also define `all_cols = list(F.columns)` for the tree models.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(results['linear'] - 6.77) < 0.05, 'linear: scaled LinearRegression on lin_cols, MAE on F_val'
print('Task 1 passed')

---
# Part 2 -- Trees can bend

## Task 2: A decision tree, depth by depth

For `depth` in `[2, 4, 6, 8, 12, None]` evaluate `DecisionTreeRegressor(max_depth=depth, random_state=0)` on `all_cols` (trees need neither scaling nor the log columns, but extra columns do not hurt). Expected MAE 7.50, 7.03, 6.83, 7.14, 8.26, 8.75. Store the depth-6 value as `results["tree"]`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(results['tree'] - 6.83) < 0.05, 'tree: DecisionTreeRegressor(max_depth=6, random_state=0) on all_cols'
print('Task 2 passed')

## Task 3: A random forest

Evaluate `RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1)` on `all_cols`, keep the fitted model as `forest` and the score as `results["forest"]`. Expected MAE about 6.58: already inside the full-mark band on the validation split.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(results['forest'] - 6.58) < 0.08, 'forest: 300 trees, random_state=0, on all_cols'
print('Task 3 passed')

## Task 4: What the forest used

Put `forest.feature_importances_` in a `pd.Series` indexed by `all_cols`, sort it and print the six largest. Expected top: `Negative` (0.217), `year` (0.201), `Recommendations` (0.135). Is the column the forest uses most one that the linear model could use at all?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert importance.index[0] == 'Negative', 'importance: pd.Series(forest.feature_importances_, index=all_cols)'
print('Task 4 passed')

## Task 5: Gradient boosting

Evaluate `GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=0)` on `all_cols` as `results["boosting"]`. Expected MAE about 6.49.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(results['boosting'] - 6.49) < 0.08, 'boosting: 100 trees, depth 3, learning rate 0.1'
print('Task 5 passed')

---
# Part 3 -- Tune it properly

## Task 6: Grid search for the boosting dials

Run `GridSearchCV` on `GradientBoostingRegressor(random_state=0)` with the grid `{"n_estimators": [100, 300], "max_depth": [2, 3, 4], "learning_rate": [0.05, 0.1]}`, `cv=3`, `scoring="neg_mean_absolute_error"`, on `F_train[all_cols], y_train` (about 12 seconds). Print `best_params_` and `-best_score_`, then evaluate `grid.best_estimator_` as `results["boosting_tuned"]`. Expected: 300 trees of depth 2 at rate 0.05, CV MAE 6.32, validation about 6.46.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert grid.best_params_ == {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 300}, 'grid: 12 combinations, 3 folds, scoring="neg_mean_absolute_error"'
print('Task 6 passed')

## Task 7: The comparison table

Turn `results` into a sorted `pd.Series` and print it, best first. How far is each model from the 6.6 line, and where did the biggest single gain come from?

In [ ]:
# Your code here


---
# Part 4 -- The submission

## Task 8: Fit on everything and write the file

Build `X_test = add_features(test)[all_cols]`, fit `GradientBoostingRegressor(**grid.best_params_, random_state=0)` on all of `F[all_cols], y`, predict, clip at 0, and save a one-column `Price` DataFrame to `submission.csv` with `index=False`. Print the shape (600, 1) and the mean predicted price (about 16.5).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
chk = pd.read_csv('submission.csv')
assert chk.shape == (600, 1) and chk['Price'].min() >= 0, 'submission.csv: one Price column, 600 rows, no negative prices'
print('Task 8 passed')

---
## Task 9: Reflect

The forest beat the line by 0.19 dollars of MAE without any log column: what does that say about which models need feature engineering most? The grid search's CV score (6.32) was better than its validation score (6.46): which of the two would you report, and why? The official solution reached 6.57 with a random forest and scored full marks. If your validation MAE is 6.46, can you be sure of full marks on the hidden test set?

*Your answers here*

---
## Conclusion

- **The line to beat**: Day 08's best linear model, MAE 6.77, rebuilt with one scorer.
- **Trees bend**: a single tree matched the line at depth 6, the forest beat it (6.58) and read its own importances, boosting beat the forest (6.49).
- **Tuning**: grid search on three dials by cross-validated MAE gave 300 shallow trees at a small learning rate, validation 6.46.
- **The submission**: the tuned model refitted on all 3,000 games, predictions clipped at zero, a file in the required format.

---

Made By **Sattam Altwaim**